# PulmoScan: finding annotation intake (not training)

Checks the **training CSVs only** for a finding/localization dataset.
The existing NORMAL/PNEUMONIA folder dataset cannot supply finding boxes.

Read `docs/FINDING_MODEL_NEXT_STEPS.md` in the project first.
Obtain your own authorized access to [VinDr-PCXR/PediCXR](https://physionet.org/content/vindr-pcxr/1.0.0/).
Use Kaggle only if your data agreement permits that processing and storage arrangement;
private visibility alone does not establish permission. Do not publish restricted inputs or outputs.
If permission is unclear, run the local annotation checker instead.

In a permitted private notebook, attach the authorized training CSVs via **Add Input**.
CPU is enough; Internet and package installation are unnecessary. No image files are read,
no test annotations are inspected, and no model is trained or replaced.
A successful check still needs image/label/split review before training.

In [ ]:
import hashlib
import types
SOURCE = '"""Inspect finding annotation CSVs without reading images or training a model.\n\nThis is an intake check, not a complete dataset audit. No image identifiers or\nexample rows are exported. Test annotations are deliberately not discovered.\n"""\n\nimport argparse\nfrom collections import Counter, defaultdict\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nfrom pathlib import Path\n\n\nCOORDINATES = (\'x_min\', \'y_min\', \'x_max\', \'y_max\')\nMISSING = {\'\', \'nan\', \'none\', \'null\'}\n\n\ndef read_csv(path, required):\n    payload = Path(path).read_bytes()\n    reader = csv.DictReader(io.StringIO(payload.decode(\'utf-8-sig\'), newline=\'\'))\n    columns = reader.fieldnames or []\n    if len(set(columns)) != len(columns) or any(not name.strip() for name in columns):\n        raise ValueError(\'CSV has empty or duplicate column names.\')\n    if not set(required).issubset(columns):\n        raise ValueError(\'Unsupported CSV schema. Required columns: \' + \', \'.join(required)\n                         + \'. Found: \' + \', \'.join(columns))\n    rows = list(reader)\n    if not rows:\n        raise ValueError(\'CSV contains no data rows.\')\n    if any(None in row or any(value is None for value in row.values()) for row in rows):\n        raise ValueError(\'CSV contains rows with missing or extra fields.\')\n    return rows, {\'sha256\': hashlib.sha256(payload).hexdigest(), \'columns\': columns, \'rows\': len(rows)}\n\n\ndef audit_annotations(annotations, image_labels):\n    boxes, box_source = read_csv(annotations, (\'image_id\', \'class_name\', *COORDINATES))\n    labels, label_source = read_csv(image_labels, (\'image_id\',))\n    errors, warnings = Counter(), Counter()\n    label_ids = {row[\'image_id\'].strip() for row in labels}\n    errors[\'empty_image_label_id\'] = sum(not row[\'image_id\'].strip() for row in labels)\n    label_ids.discard(\'\')\n    warnings[\'additional_image_label_rows\'] = len(labels) - len(label_ids)\n    if \'rad_id\' not in box_source[\'columns\']:\n        warnings[\'reviewer_column_absent\'] = 1\n    classes, class_images, class_ids = Counter(), defaultdict(set), defaultdict(set)\n    class_names_by_id = defaultdict(set)\n    positive_ids, background_ids, annotated_ids, seen = set(), set(), set(), set()\n    for row in boxes:\n        image_id, name = row[\'image_id\'].strip(), row[\'class_name\'].strip()\n        if not image_id or not name:\n            errors[\'empty_annotation_id_or_class\'] += 1\n            continue\n        annotated_ids.add(image_id)\n        if image_id not in label_ids:\n            errors[\'annotation_image_missing_from_image_labels\'] += 1\n        values = [row[key].strip() for key in COORDINATES]\n        if name.casefold() == \'no finding\':\n            background_ids.add(image_id)\n            # Releases may encode the background sentinel as empty/NaN or zeros.\n            present = [value for value in values if value.casefold() not in MISSING]\n            try:\n                if any(not math.isfinite(float(value)) or float(value) != 0 for value in present):\n                    errors[\'background_row_has_coordinates\'] += 1\n            except ValueError:\n                errors[\'background_row_has_coordinates\'] += 1\n            continue\n        positive_ids.add(image_id)\n        try:\n            coordinates = tuple(float(value) for value in values)\n        except ValueError:\n            errors[\'invalid_box_coordinates\'] += 1\n            continue\n        x_min, y_min, x_max, y_max = coordinates\n        if (not all(math.isfinite(value) for value in coordinates)\n                or min(coordinates) < 0 or x_max <= x_min or y_max <= y_min):\n            errors[\'invalid_box_coordinates\'] += 1\n            continue\n        reviewer = row.get(\'rad_id\', \'\').strip()\n        class_id = row.get(\'class_id\', \'\').strip()\n        if class_id:\n            class_ids[name].add(class_id)\n            class_names_by_id[class_id].add(name)\n        key = (image_id, name, reviewer, coordinates)\n        if key in seen:\n            warnings[\'duplicate_box_rows\'] += 1\n            continue\n        seen.add(key)\n        classes[name] += 1\n        class_images[name].add(image_id)\n    errors[\'background_and_positive_same_image\'] = len(background_ids & positive_ids)\n    errors[\'class_name_has_multiple_ids\'] = sum(len(values) > 1 for values in class_ids.values())\n    errors[\'class_id_has_multiple_names\'] = sum(len(values) > 1 for values in class_names_by_id.values())\n    if not classes:\n        errors[\'no_valid_finding_boxes\'] = 1\n    errors = {key: count for key, count in sorted(errors.items()) if count}\n    warnings = {key: count for key, count in sorted(warnings.items()) if count}\n    return {\n        \'schema_version\': 1, \'stage\': \'annotation_intake_only\',\n        \'status\': \'blocked\' if errors else \'needs_review\', \'training_ready\': False,\n        \'inputs\': {\'annotations\': box_source, \'image_labels\': label_source},\n        \'counts\': {\n            \'image_label_images\': len(label_ids), \'annotated_images\': len(annotated_ids),\n            \'images_with_finding_rows\': len(positive_ids), \'explicit_no_finding_images\': len(background_ids),\n            \'images_without_annotation_rows\': len(label_ids - annotated_ids),\n            \'unique_valid_boxes\': sum(classes.values()),\n        },\n        \'candidate_classes\': [\n            {\'name\': name, \'boxes\': classes[name], \'images\': len(class_images[name]),\n             \'source_class_ids\': sorted(class_ids[name])}\n            for name in sorted(classes)\n        ],\n        \'errors\': errors, \'warnings\': warnings,\n        \'limitations\': [\n            \'Only CSV structure, class counts and box geometry were checked.\',\n            \'Image-level label values and clinical meanings have not been validated.\',\n            \'Missing box rows do not establish a normal or fully annotated image.\',\n            \'No DICOM decoding, dimensions, orientation, pixel hashes or duplicate-image audit was performed.\',\n            \'Box coordinates must still be verified against source image dimensions and previews.\',\n            \'No train/validation/calibration split or cross-split patient/duplicate check was performed.\',\n            \'No test-set metrics, training, model promotion or clinical validation was performed.\',\n        ],\n    }\n\n\ndef discover_training_root(search_root):\n    candidates = sorted({path.parent.resolve() for path in Path(search_root).rglob(\'annotations_train.csv\')\n                         if (path.parent / \'image_labels_train.csv\').is_file()})\n    if len(candidates) != 1:\n        raise ValueError(\'Expected one dataset with annotations_train.csv and image_labels_train.csv; \'\n                         f\'found {len(candidates)}. Set DATA_ROOT explicitly. The binary dataset is insufficient.\')\n    return candidates[0]\n\n\ndef save_report(report, destination):\n    # Never overwrite an earlier audit or an accidentally selected input file.\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with destination.open(\'x\', encoding=\'utf-8\') as stream:\n        stream.write(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + \'\\n\')\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--annotations\', type=Path, required=True, help=\'Training bounding-box CSV\')\n    parser.add_argument(\'--image-labels\', type=Path, required=True, help=\'Training image-level CSV\')\n    parser.add_argument(\'--output\', type=Path, required=True, help=\'New report path; never overwritten\')\n    args = parser.parse_args()\n    try:\n        report = audit_annotations(args.annotations, args.image_labels)\n        save_report(report, args.output)\n    except (ValueError, OSError, UnicodeError, csv.Error) as exc:\n        parser.exit(2, f\'Annotation check failed: {exc}\\n\')\n    print(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False))\n    raise SystemExit(2 if report[\'errors\'] else 0)\n\n\nif __name__ == \'__main__\':\n    main()\n'
if hashlib.sha256(SOURCE.encode('utf-8')).hexdigest() != '677c99eeefc254ee73e264775b0d40a643e91ce9732fcb96f142da589551dd6d':
    raise ValueError('Notebook source checksum failed. Re-import an intact notebook.')
checker = types.ModuleType('pulmoscan_finding_intake')
exec(compile(SOURCE, '<embedded annotation checker>', 'exec'), checker.__dict__)
print('Loaded annotation checker; no training or installations.')

In [ ]:
from pathlib import Path

# Directory directly containing BOTH training CSVs. Leave None if only one matches.
DATA_ROOT = None
# Example: DATA_ROOT = '/kaggle/input/your-authorized-input/dataset'

root = (Path(DATA_ROOT) if DATA_ROOT is not None
        else checker.discover_training_root(Path('/kaggle/input')))
report = checker.audit_annotations(root / 'annotations_train.csv', root / 'image_labels_train.csv')

In [ ]:
from datetime import datetime, timezone
import json
from uuid import uuid4

output = Path('/kaggle/working') / (
    'pulmoscan_finding_intake_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    + '_' + uuid4().hex[:8] + '.json')
checker.save_report(report, output)
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Report:', output.name)
if report['errors']:
    raise ValueError('Annotation errors found. Review the saved report; do not start training.')
print('Intake completed. Image, label and split review still required. Training is not ready.')